In [6]:
#!pip install pandas

In [2]:
import pandas as pd
import datetime


In [4]:
def generate_raw_data():
    """Generates a sample list of raw sales transaction records."""
    data = [
        {
            "transaction_id": "TRX-20250730-001", "order_timestamp": "2025-07-30 11:15:42",
            "customer_id": 1845, "customer_name": "Rohan Sharma", "customer_city": "Mumbai", "customer_state": "Maharashtra",
            "product_sku": "LAP-MAC-M3-SLV", "product_name": "MacBook Air M3 13-inch", "product_category": "Laptops", "product_brand": "Apple",
            "store_id": "MUM-01", "store_location": "Bandra, Mumbai",
            "quantity_sold": 1, "unit_price": 115000.00
        },
        {
            "transaction_id": "TRX-20250730-002", "order_timestamp": "2025-07-30 13:22:01",
            "customer_id": 2109, "customer_name": "Priya Singh", "customer_city": "Bengaluru", "customer_state": "Karnataka",
            "product_sku": "PHN-SAM-S24-BLK", "product_name": "Samsung Galaxy S24", "product_category": "Smartphones", "product_brand": "Samsung",
            "store_id": "BLR-02", "store_location": "Koramangala, Bengaluru",
            "quantity_sold": 1, "unit_price": 79999.00
        },
        {
            "transaction_id": "TRX-20250731-003", "order_timestamp": "2025-07-31 09:05:18",
            "customer_id": 1845, "customer_name": "Rohan Sharma", "customer_city": "Mumbai", "customer_state": "Maharashtra",
            "product_sku": "ACC-LOG-MXM3", "product_name": "Logitech MX Master 3S Mouse", "product_category": "Accessories", "product_brand": "Logitech",
            "store_id": "MUM-01", "store_location": "Bandra, Mumbai",
            "quantity_sold": 1, "unit_price": 9500.00
        },
        {
            "transaction_id": "TRX-20250731-004", "order_timestamp": "2025-07-31 15:45:50",
            "customer_id": 3541, "customer_name": "Anjali Verma", "customer_city": "Delhi", "customer_state": "Delhi",
            "product_sku": "LAP-MAC-M3-SLV", "product_name": "MacBook Air M3 13-inch", "product_category": "Laptops", "product_brand": "Apple",
            "store_id": "DEL-01", "store_location": "Connaught Place, Delhi",
            "quantity_sold": 2, "unit_price": 115000.00
        },
        {
            "transaction_id": "TRX-20250801-005", "order_timestamp": "2025-08-01 18:12:30",
            "customer_id": 2109, "customer_name": "Priya Singh", "customer_city": "Bengaluru", "customer_state": "Karnataka",
            "product_sku": "LAP-DELL-XPS15", "product_name": "Dell XPS 15", "product_category": "Laptops", "product_brand": "Dell",
            "store_id": "BLR-02", "store_location": "Koramangala, Bengaluru",
            "quantity_sold": 1, "unit_price": 145000.00
        }
    ]
    return data


In [5]:
def load_dimensions(raw_data):
    """
    Processes raw data to create and populate dimension tables.
    Generates surrogate keys for each dimension.
    """
    # --- Dimension 1: Customer ---
    # Extract unique customers based on customer_id
    customers = {d['customer_id']: d for d in raw_data}.values()
    dim_customer_df = pd.DataFrame(customers)[['customer_id', 'customer_name', 'customer_city', 'customer_state']]
    dim_customer_df.drop_duplicates(subset=['customer_id'], inplace=True)
    dim_customer_df.insert(0, 'Customer_Key', range(101, 101 + len(dim_customer_df))) # Surrogate Key

    # --- Dimension 2: Product ---
    # Extract unique products based on product_sku
    products = {d['product_sku']: d for d in raw_data}.values()
    dim_product_df = pd.DataFrame(products)[['product_sku', 'product_name', 'product_category', 'product_brand']]
    dim_product_df.drop_duplicates(subset=['product_sku'], inplace=True)
    dim_product_df.insert(0, 'Product_Key', range(5001, 5001 + len(dim_product_df))) # Surrogate Key

    # --- Dimension 3: Store ---
    # Extract unique stores based on store_id
    stores = {d['store_id']: d for d in raw_data}.values()
    dim_store_df = pd.DataFrame(stores)[['store_id', 'store_location']]
    dim_store_df.drop_duplicates(subset=['store_id'], inplace=True)
    dim_store_df.insert(0, 'Store_Key', range(801, 801 + len(dim_store_df))) # Surrogate Key

    # --- Dimension 4: Date ---
    # Extract unique dates from the timestamp and create a rich date dimension
    dates = {pd.to_datetime(d['order_timestamp']).date() for d in raw_data}
    dim_date_df = pd.DataFrame(sorted(list(dates)), columns=['Full_Date'])
    dim_date_df['Full_Date'] = pd.to_datetime(dim_date_df['Full_Date'])
    dim_date_df.insert(0, 'Date_Key', dim_date_df['Full_Date'].dt.strftime('%Y%m%d').astype(int)) # Surrogate Key (meaningful)
    dim_date_df['Day_Of_Week'] = dim_date_df['Full_Date'].dt.day_name()
    dim_date_df['Day_Of_Month'] = dim_date_df['Full_Date'].dt.day
    dim_date_df['Month_Name'] = dim_date_df['Full_Date'].dt.month_name()
    dim_date_df['Quarter'] = 'Q' + dim_date_df['Full_Date'].dt.quarter.astype(str)
    dim_date_df['Year'] = dim_date_df['Full_Date'].dt.year

    # Return a dictionary of dimension dataframes for easy access
    return {
        'customer': dim_customer_df,
        'product': dim_product_df,
        'store': dim_store_df,
        'date': dim_date_df
    }


In [7]:
def load_fact_table(raw_data, dims):
    """
    Loads the fact table by looking up surrogate keys from the dimension tables.
    """
    fact_sales_records = []

    # Create mapping dictionaries for faster lookups
    # Maps natural key (e.g., customer_id) to surrogate key (e.g., Customer_Key)
    customer_key_map = dims['customer'].set_index('customer_id')['Customer_Key'].to_dict()
    product_key_map = dims['product'].set_index('product_sku')['Product_Key'].to_dict()
    store_key_map = dims['store'].set_index('store_id')['Store_Key'].to_dict()
    date_key_map = dims['date'].set_index('Full_Date')['Date_Key'].to_dict()

    for record in raw_data:
        # --- Transformation and Lookup Step ---
        # Get the date part of the timestamp
        order_date = pd.to_datetime(record['order_timestamp']).normalize()

        # Look up the surrogate key for each dimension
        customer_key = customer_key_map.get(record['customer_id'])
        product_key = product_key_map.get(record['product_sku'])
        store_key = store_key_map.get(record['store_id'])
        date_key = date_key_map.get(order_date)
        
        # --- Measures Calculation ---
        quantity = record['quantity_sold']
        unit_price = record['unit_price']
        total_sales = quantity * unit_price

        # Append the processed record to our list
        fact_sales_records.append({
            'Date_Key': date_key,
            'Customer_Key': customer_key,
            'Product_Key': product_key,
            'Store_Key': store_key,
            'Transaction_ID_Degenerate': record['transaction_id'], # Degenerate Dimension
            'Quantity_Sold': quantity,
            'Unit_Price': unit_price,
            'Total_Sales_Amount': total_sales
        })

    return pd.DataFrame(fact_sales_records)


In [8]:
# --- Main ETL Execution ---
if __name__ == "__main__":
    # 1. EXTRACT: Get the raw data from the source
    raw_sales_data = generate_raw_data()
    print("--- 1. Sample Raw Transactional Record ---")
    print(raw_sales_data[0])
    print("\n" + "="*50 + "\n")

    # 2. TRANSFORM and LOAD Dimensions: Process raw data and create dimension tables
    print("--- 2. Loading Dimension Tables ---")
    dimensions = load_dimensions(raw_sales_data)

    print("\n--- DimCustomer ---")
    print(dimensions['customer'].to_string())

    print("\n--- DimProduct ---")
    print(dimensions['product'].to_string())

    print("\n--- DimStore ---")
    print(dimensions['store'].to_string())

    print("\n--- DimDate ---")
    print(dimensions['date'].to_string())
    print("\n" + "="*50 + "\n")

    # 3. TRANSFORM and LOAD Fact Table: Use dimensions to create the fact table
    print("--- 3. Loading Fact Table ---")
    fact_sales_df = load_fact_table(raw_sales_data, dimensions)
    print("\n--- FactSales ---")
    print(fact_sales_df.to_string())


--- 1. Sample Raw Transactional Record ---
{'transaction_id': 'TRX-20250730-001', 'order_timestamp': '2025-07-30 11:15:42', 'customer_id': 1845, 'customer_name': 'Rohan Sharma', 'customer_city': 'Mumbai', 'customer_state': 'Maharashtra', 'product_sku': 'LAP-MAC-M3-SLV', 'product_name': 'MacBook Air M3 13-inch', 'product_category': 'Laptops', 'product_brand': 'Apple', 'store_id': 'MUM-01', 'store_location': 'Bandra, Mumbai', 'quantity_sold': 1, 'unit_price': 115000.0}


--- 2. Loading Dimension Tables ---

--- DimCustomer ---
   Customer_Key  customer_id customer_name customer_city customer_state
0           101         1845  Rohan Sharma        Mumbai    Maharashtra
1           102         2109   Priya Singh     Bengaluru      Karnataka
2           103         3541  Anjali Verma         Delhi          Delhi

--- DimProduct ---
   Product_Key      product_sku                 product_name product_category product_brand
0         5001   LAP-MAC-M3-SLV       MacBook Air M3 13-inch         